In [40]:
import numpy as np
import pandas as pd
import os

In [14]:
# Configuration

config = {
    "n_samples"      : 1000,   # total number of telemetry readings
    "fault_rate"     : 0.15,   # 15% of samples will be faults
    "fault_duration" : 20,     # faults last 20 consecutive samples (like real anomaly events)
    "noise_level"    : 0.05,   # gaussian noise added to all sensors
    "random_seed"    : 42      # for reproducibility
}

In [16]:
# Normal sensor ranges

normal_ranges = {
    "battery_voltage"    : (24.0, 28.0),   # Volts
    "solar_current"      : (1.5,  3.5),    # Amperes
    "panel_temp"         : (-10.0, 60.0),  # Celsius
    "onboard_temp"       : (15.0,  35.0),  # Celsius
    "attitude_error"     : (0.0,   0.5),   # Degrees
    "reaction_wheel_rpm" : (1000.0, 5000.0) # RPM
}

In [18]:
# Fault signatures
# This defines what each sensor looks like during a fault. 
# These are the values the generator will inject into the fault windows.

fault_signatures = {
    "battery_voltage"    : (18.0, 21.0),   # Volts — drops dangerously low
    "solar_current"      : (0.0,  0.2),    # Amperes — near zero, panel failure
    "panel_temp"         : (80.0, 95.0),   # Celsius — thermal runaway
    "onboard_temp"       : (50.0, 65.0),   # Celsius — overheating
    "attitude_error"     : (2.0,  5.0),    # Degrees — attitude control lost
    "reaction_wheel_rpm" : (0.0,  100.0)   # RPM — wheel failure
}

In [26]:
# Generate normal data

np.random.seed(config["random_seed"])

# create empty DataFrame with n_samples rows
df = pd.DataFrame()

# fill each sensor column with normal readings
for sensor, (low, high) in normal_ranges.items():
    df[sensor] = np.random.uniform(low, high, config["n_samples"])

# add gaussian noise to every sensor
for sensor in normal_ranges.keys():
    df[sensor] += np.random.normal(0, config["noise_level"], config["n_samples"])

# all samples start as nominal (label = 0)
df["label"] = 0

print(df.shape)
print(df.head())

(1000, 7)
   battery_voltage  solar_current  panel_temp  onboard_temp  attitude_error  \
0        25.445854       1.921408    8.326763     28.433852        0.300639   
1        27.826528       2.574280    7.254491     30.933718        0.308694   
2        26.942523       3.217188   53.468568     20.015243        0.301611   
3        26.442526       2.950393    7.493718     27.413894        0.045106   
4        24.584530       3.142090    9.077066     26.570436        0.084982   

   reaction_wheel_rpm  label  
0         2574.605562      0  
1         2893.680380      0  
2         4418.257454      0  
3         2360.010365      0  
4         4478.619665      0  


In [28]:
#  Inject faults 

n_faults = int(config["n_samples"] * config["fault_rate"])
n_windows = n_faults // config["fault_duration"]

for _ in range(n_windows):
    # pick a random start point for this fault window
    start = np.random.randint(0, config["n_samples"] - config["fault_duration"])
    end   = start + config["fault_duration"]
    
    # overwrite every sensor in that window with fault values
    for sensor, (low, high) in fault_signatures.items():
        df.loc[start:end, sensor] = np.random.uniform(low, high, config["fault_duration"] + 1)
    
    # mark those rows as fault
    df.loc[start:end, "label"] = 1

print("Total fault samples  :", df["label"].sum())
print("Total nominal samples:", (df["label"] == 0).sum())
print("Fault rate           :", df["label"].sum() / len(df))

Total fault samples  : 124
Total nominal samples: 876
Fault rate           : 0.124


In [42]:
# Save to CSV

# create the output folder if it doesn't exist
os.makedirs("../data/synthetic", exist_ok=True)

# save
df.to_csv("../data/synthetic/telemetry.csv", index=False)

print("Saved to data/synthetic/telemetry.csv")
print("Shape:", df.shape)

Saved to data/synthetic/telemetry.csv
Shape: (1000, 7)


In [44]:
# Verify the saved file

# read it back and confirm it matches what we generated
df_check = pd.read_csv("../data/synthetic/telemetry.csv")

print("Shape          :", df_check.shape)
print("Columns        :", list(df_check.columns))
print("Fault samples  :", df_check["label"].sum())
print("Nominal samples:", (df_check["label"] == 0).sum())
print("\nFirst 3 rows:")
print(df_check.head(3))

Shape          : (1000, 7)
Columns        : ['battery_voltage', 'solar_current', 'panel_temp', 'onboard_temp', 'attitude_error', 'reaction_wheel_rpm', 'label']
Fault samples  : 124
Nominal samples: 876

First 3 rows:
   battery_voltage  solar_current  panel_temp  onboard_temp  attitude_error  \
0        25.445854       1.921408    8.326763     28.433852        0.300639   
1        27.826528       2.574280    7.254491     30.933718        0.308694   
2        26.942523       3.217188   53.468568     20.015243        0.301611   

   reaction_wheel_rpm  label  
0         2574.605562      0  
1         2893.680380      0  
2         4418.257454      0  


In [46]:
# To verify the .py file works as a module

import sys
sys.path.append("../src")

from data_generator import generate, save

df = generate()
print("Import successful")
print("Shape:", df.shape)
print("Fault samples:", df["label"].sum())


Import successful
Shape: (1000, 7)
Fault samples: 124
